# Análisis Exploratorio y Modelado Predictivo del Censo Operativo

## Pipeline de Analítica para la Distribución de Gas LP

**Autor:** Ivan Octavio Ortiz Gallardo  
**Entorno de desarrollo:** Databricks | Apache Spark | PySpark | Python

Este notebook documenta el análisis exploratorio, la preparación de variables, el control estadístico y el modelado predictivo realizado sobre el conjunto de datos consolidado del censo operativo.

La información utilizada proviene de una tabla administrada en Apache Spark, previamente generada durante la etapa de integración y transformación mediante SQL.


## Objetivo

Desarrollar un flujo de análisis y modelado predictivo a partir del conjunto de datos consolidado del censo operativo, con el propósito de identificar patrones, evaluar el comportamiento del proceso y generar proyecciones sobre las entregas.

## Contenido

1. Preparación y validación de datos
2. Análisis Exploratorio de Datos (EDA)
3. Análisis descriptivo
4. Limpieza e ingeniería de variables
5. Control estadístico del proceso
6. Modelado predictivo con Prophet
7. Integración de datos reales y pronosticados
8. Validación del modelo
9. Persistencia de resultados

> Por motivos de confidencialidad, los nombres de tablas y rutas utilizados en el entorno institucional han sido anonimizados.


In [0]:

# Configuración e importación de librerías


%matplotlib inline

# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Estadística
from scipy import stats
from scipy.stats import zscore

# Modelado de series de tiempo
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric


## 1. Preparación de datos

El conjunto de datos utilizado en este notebook corresponde a la información consolidada del censo operativo.

La información se recupera desde una tabla administrada en Apache Spark dentro del entorno Databricks. Esta tabla constituye el punto de partida para las etapas de exploración, transformación, control estadístico y modelado predictivo.

> **Confidencialidad:** los nombres de tablas y rutas del entorno institucional fueron anonimizados para su publicación.


In [0]:

# Carga del conjunto de datos desde Apache Spark


TABLE_NAME = "analytics.balance_censo_dataset"

# Recuperación de la tabla consolidada
df_spark = spark.table(TABLE_NAME)

# Vista preliminar de los registros
df_spark.show(5)

# Verificación de la estructura del conjunto de datos
df_spark.printSchema()


In [0]:

# Conversión de Spark DataFrame a Pandas DataFrame


# Conversión a Pandas para realizar el análisis exploratorio
df_balance_censo = df_spark.toPandas()

# Vista preliminar
df_balance_censo.head()


## 2. Análisis Exploratorio de Datos (EDA)

En esta etapa se revisa la estructura y calidad inicial del conjunto de datos antes de realizar transformaciones y modelado.

Se consideran los tipos de datos, dimensiones, estadística descriptiva y presencia de valores faltantes.


In [0]:

# Estructura del conjunto de datos


# Tipos de datos de las variables
df_balance_censo.dtypes

# Información general: registros no nulos y uso de memoria
df_balance_censo.info()


### 2.1 Análisis descriptivo

Se obtiene un resumen estadístico de las variables numéricas y categóricas para conocer su comportamiento general, dispersión y posibles anomalías.


In [0]:

# Estadística descriptiva


# Resumen de variables numéricas
df_balance_censo.describe()

# Resumen de variables categóricas
df_balance_censo.describe(include=["object"])


## 3. Limpieza y transformación de datos

Esta etapa prepara las variables necesarias para el análisis temporal y el modelado predictivo.

Se verifica especialmente el formato de las fechas y la presencia de valores faltantes.


### 3.1 Conversión de variables de fecha

La variable `Fecha_Conv` se convierte a formato `datetime` para permitir operaciones temporales, agrupaciones por periodo y su utilización como eje temporal en Prophet.


In [0]:

# Conversión de la variable de fecha


df_balance_censo["Fecha_Conv"] = pd.to_datetime(
    df_balance_censo["Fecha_Conv"],
    errors="coerce"
)


In [0]:

# Validación de la conversión de fechas


print("Tipo de dato:", df_balance_censo["Fecha_Conv"].dtype)
print("Fechas no convertidas:", df_balance_censo["Fecha_Conv"].isna().sum())


### 3.2 Valores faltantes

Se cuantifican los valores nulos de cada variable para identificar posibles problemas de calidad y documentar las columnas que requieren consideración durante las etapas posteriores.


In [0]:

# Conteo de valores faltantes por columna


df_balance_censo.isnull().sum().sort_values(ascending=False)


### 3.3 Verificación general del conjunto de datos

Se revisa nuevamente la estructura del DataFrame después de la conversión de fechas y de las validaciones iniciales.


In [0]:

# Información general del DataFrame


df_balance_censo.info()


In [0]:

# 4. Ingeniería de variables


# Normalización de las variables utilizadas en el modelado
df_balance_censo["ENTREGAS"] = pd.to_numeric(
    df_balance_censo["ENTREGAS"],
    errors="coerce"
)

df_balance_censo["RECIBOS"] = pd.to_numeric(
    df_balance_censo["RECIBOS"],
    errors="coerce"
)

# Conversión de unidades a miles de unidades base
df_balance_censo["entregas_mdb"] = df_balance_censo["ENTREGAS"] / 1000
df_balance_censo["recibos_mdb"] = df_balance_censo["RECIBOS"] / 1000


## 5. Control estadístico del proceso

Se aplica un gráfico de control tipo Shewhart sobre la variable de entregas.

Para reducir la variabilidad diaria y facilitar la identificación de cambios en el comportamiento del proceso, se utilizan promedios semanales como serie de análisis.

Los límites se calculan a partir de la media y la desviación estándar de la serie semanal.


In [0]:

# Gráfico de control Shewhart para ENTREGAS


# Asegurar el formato temporal
df_balance_censo["Fecha_Conv"] = pd.to_datetime(
    df_balance_censo["Fecha_Conv"],
    errors="coerce"
)

# Promedio semanal de ENTREGAS
df_semanal = (
    df_balance_censo
    .groupby(pd.Grouper(key="Fecha_Conv", freq="W"))["ENTREGAS"]
    .mean()
    .reset_index()
    .sort_values("Fecha_Conv")
)

# Parámetros del proceso
media = df_semanal["ENTREGAS"].mean()
std = df_semanal["ENTREGAS"].std()

# Límites de control
df_shewhart = df_semanal.copy()
df_shewhart["Media"] = media
df_shewhart["+1σ"] = media + std
df_shewhart["-1σ"] = media - std
df_shewhart["+2σ"] = media + 2 * std
df_shewhart["-2σ"] = media - 2 * std
df_shewhart["+3σ"] = media + 3 * std
df_shewhart["-3σ"] = media - 3 * std

# Visualización
plt.figure(figsize=(12, 6))

plt.plot(
    df_semanal["Fecha_Conv"],
    df_semanal["ENTREGAS"],
    marker="o",
    label="Promedio semanal"
)

plt.axhline(media, linestyle="-", linewidth=2, label="Media")
plt.axhline(media + std, linestyle="--", label="+1σ")
plt.axhline(media - std, linestyle="--", label="-1σ")
plt.axhline(media + 2 * std, linestyle="-.", label="+2σ")
plt.axhline(media - 2 * std, linestyle="-.", label="-2σ")
plt.axhline(media + 3 * std, linestyle=":", linewidth=1.5, label="+3σ")
plt.axhline(media - 3 * std, linestyle=":", linewidth=1.5, label="-3σ")

plt.title("Gráfico de Control Shewhart - ENTREGAS (Promedio semanal)")
plt.xlabel("Fecha")
plt.ylabel("ENTREGAS")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

df_shewhart.head()


In [0]:

# Persistencia del resultado del control estadístico


df_spark_shewhart = spark.createDataFrame(df_shewhart)

df_spark_shewhart.write \
    .mode("overwrite") \
    .saveAsTable("analytics.shewhart_entregas")


### 5.1 Índices de capacidad y centramiento del proceso

Se calculan los índices `Cp` y `Cpk`, el centramiento del proceso (`K`) y una prueba de normalidad.

La clasificación y las recomendaciones se mantienen conforme a las reglas definidas originalmente para el proyecto.


In [0]:

# Cálculo de capacidad y centramiento del proceso


datos = pd.to_numeric(
    df_shewhart["ENTREGAS"],
    errors="coerce"
).dropna()

# Límites de control
LSE = df_shewhart["+3σ"].iloc[-1]
LIE = df_shewhart["-3σ"].iloc[-1]

def generar_recomendaciones(evaluacion, Cpk, K):
    """Genera recomendaciones a partir de la evaluación del proceso."""
    recomendaciones = []

    if not evaluacion["CAPAZ"]:
        if Cpk < 1.0:
            recomendaciones.extend([
                "Reducir variabilidad del proceso",
                "Inspección 100% del producto",
                "Parada del proceso para análisis",
                "Ajuste/recalibración completa",
                "Asignación de equipo de mejora",
                "Revisión de especificaciones con cliente"
            ])
        elif Cpk < 1.33:
            recomendaciones.extend([
                "Mejorar capacidad del proceso",
                "Implementar SPC (Control Estadístico de Proceso)",
                "Revisar y ajustar parámetros críticos",
                "Capacitación operativa reforzada",
                "Aumentar frecuencia de muestreo",
                "Análisis de capacidad por característica",
                "Mejora de herramientas y equipos",
                "Estandarización de métodos"
            ])

    if not evaluacion["BIEN_CENTRADO"]:
        recomendaciones.append("Ajustar centramiento del proceso")

    if not evaluacion["NORMAL"]:
        recomendaciones.append("Investigar causas de no-normalidad")

    return recomendaciones

def evaluar_estado_proceso(datos, LSE, LIE, alpha=0.05):
    """Evalúa capacidad, centramiento, normalidad y estabilidad del proceso."""

    media = np.mean(datos)
    std = np.std(datos, ddof=1)

    objetivo = (LSE + LIE) / 2

    Cp = (LSE - LIE) / (6 * std)
    Cpk = min(
        (LSE - media) / (3 * std),
        (media - LIE) / (3 * std)
    )

    K = abs((media - objetivo) / ((LSE - LIE) / 2)) * 100

    _, p_normal = stats.normaltest(datos)

    evaluacion = {
        "CAPAZ": Cpk >= 1.33,
        "ACEPTABLE": Cpk >= 1.0,
        "BIEN_CENTRADO": K <= 20,
        "NORMAL": p_normal > alpha,
        "ESTABLE": abs(Cp - Cpk) < 0.3
    }

    criterios_cumplidos = sum(evaluacion.values())
    estado_general = (
        "CONTROLADO"
        if criterios_cumplidos >= 4
        else "NO CONTROLADO"
    )

    return {
        "metricas": {
            "Cp": Cp,
            "Cpk": Cpk,
            "K": K,
            "Media": media,
            "Std": std
        },
        "evaluacion": evaluacion,
        "estado": estado_general,
        "recomendaciones": generar_recomendaciones(
            evaluacion,
            Cpk,
            K
        )
    }

resultado = evaluar_estado_proceso(datos, LSE, LIE)

print("=== RESUMEN DEL PROCESO ===")
print(f"Estado general del proceso: {resultado['estado']}")
print(f"Cp: {resultado['metricas']['Cp']:.3f}")
print(f"Cpk: {resultado['metricas']['Cpk']:.3f}")
print(f"Desviación estándar: {resultado['metricas']['Std']:.3f}")
print(f"Centramiento (K): {resultado['metricas']['K']:.1f}%")

print("
Recomendaciones:")
for i, rec in enumerate(resultado["recomendaciones"], 1):
    print(f"{i}. {rec}")


## 6. Modelado predictivo de entregas

Se utiliza Prophet para generar pronósticos diarios de la variable `entregas_mdb`.

Se consideran dos niveles de análisis:

- **Pronóstico por terminal y gestión:** un modelo independiente para cada combinación de `DescripciondeTerminal` y `GestiondeTerminal`.
- **Pronóstico general:** un modelo adicional sobre el conjunto preparado para obtener una referencia global.

El horizonte utilizado en ambos casos es de 30 días.


In [0]:

# Función para entrenar un modelo Prophet por grupo


def train_and_forecast(group, periods=30, freq="D"):
    """Entrena un modelo Prophet para una terminal y gestión."""

    df_p = (
        group.rename(columns={
            "Fecha_Conv": "ds",
            "entregas_mdb": "y"
        })[["ds", "y"]]
        .dropna(subset=["ds", "y"])
        .sort_values("ds")
    )

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False
    )

    model.fit(df_p)

    future = model.make_future_dataframe(
        periods=periods,
        freq=freq
    )

    forecast_group = model.predict(future)[[
        "ds",
        "yhat",
        "yhat_lower",
        "yhat_upper",
        "trend",
        "trend_lower",
        "trend_upper"
    ]]

    forecast_group["DescripciondeTerminal"] = (
        group["DescripciondeTerminal"].iloc[0]
    )
    forecast_group["GestiondeTerminal"] = (
        group["GestiondeTerminal"].iloc[0]
    )

    return forecast_group


In [0]:

# Generación de pronósticos por terminal y gestión


pronosticos = []

for (terminal, gestion), group in df_balance_censo.groupby(
    ["DescripciondeTerminal", "GestiondeTerminal"],
    dropna=False
):
    forecast_group = train_and_forecast(group)

    # Garantizar la identificación del grupo
    forecast_group["DescripciondeTerminal"] = terminal
    forecast_group["GestiondeTerminal"] = gestion

    pronosticos.append(forecast_group)

df_forecast = pd.concat(
    pronosticos,
    ignore_index=True
)

print(df_forecast.head())


In [0]:
# Últimos registros del pronóstico agrupado
df_forecast.tail(10)


In [0]:

# Preparación del conjunto para el pronóstico general


df_prophet_entregas = df_balance_censo[
    ["Fecha_Conv", "entregas_mdb"]
].copy()

# Formato de fecha requerido por Prophet
df_prophet_entregas["ds"] = pd.to_datetime(
    df_prophet_entregas["Fecha_Conv"],
    errors="coerce"
)

# Variable objetivo
df_prophet_entregas["y"] = pd.to_numeric(
    df_prophet_entregas["entregas_mdb"],
    errors="coerce"
)

df_prophet_entregas = df_prophet_entregas[
    ["ds", "y"]
].dropna().sort_values("ds")

df_prophet_entregas.head(10)


In [0]:
# Validación del conjunto preparado para Prophet
df_prophet_entregas.head(10)


In [0]:
# Últimas observaciones del conjunto utilizado para el modelo general
df_prophet_entregas.tail()


In [0]:

# Entrenamiento del modelo Prophet general


m = Prophet()

m.fit(df_prophet_entregas)


In [0]:

# Generación del horizonte de pronóstico


future = m.make_future_dataframe(
    periods=30,
    freq="D"
)

forecast = m.predict(future)


In [0]:
# Últimas observaciones del pronóstico general
forecast[
    [
        "ds",
        "yhat",
        "yhat_lower",
        "yhat_upper",
        "trend",
        "trend_lower",
        "trend_upper"
    ]
].tail(10)


In [0]:

# Visualización del pronóstico general


Pronostico = m.plot(forecast)

plt.title("Pronóstico general de entregas con Prophet")
plt.xlabel("Fecha")
plt.ylabel("Entregas (MDB)")
plt.tight_layout()
plt.show()


In [0]:

# Visualización de componentes del modelo


Componentes = m.plot_components(forecast)
plt.tight_layout()
plt.show()


## 7. Integración de datos reales y pronosticados

Se construye un conjunto final que combina los valores reales de entregas con los valores generados por el modelo de pronóstico por terminal y gestión.

Este conjunto será utilizado posteriormente para el análisis, visualización y persistencia de resultados.


In [0]:

# Preparación de los valores reales


df_entregas = df_balance_censo[
    [
        "Fecha_Conv",
        "DescripciondeTerminal",
        "GestiondeTerminal",
        "entregas_mdb"
    ]
].copy()

df_entregas.head(10)


In [0]:

# Integración del pronóstico con los valores reales


# Renombrar ds para utilizarlo como llave de fecha
df_forecast_merge = df_forecast.rename(
    columns={"ds": "Fecha"}
).copy()

# Integración mediante fecha, terminal y gestión
df_merge = df_forecast_merge.merge(
    df_balance_censo[
        [
            "Fecha_Conv",
            "DescripciondeTerminal",
            "GestiondeTerminal",
            "entregas_mdb"
        ]
    ],
    left_on=[
        "Fecha",
        "DescripciondeTerminal",
        "GestiondeTerminal"
    ],
    right_on=[
        "Fecha_Conv",
        "DescripciondeTerminal",
        "GestiondeTerminal"
    ],
    how="left"
)

# Eliminar la llave duplicada de fecha
df_merge.drop(
    columns=["Fecha_Conv"],
    inplace=True
)

# Reordenar columnas principales
cols_ordenadas = [
    "Fecha",
    "DescripciondeTerminal",
    "GestiondeTerminal",
    "entregas_mdb"
]

otras_cols = [
    c for c in df_merge.columns
    if c not in cols_ordenadas
]

df_merge = df_merge[
    cols_ordenadas + otras_cols
]

print(df_merge.head())


In [0]:
# Inspección del conjunto integrado
df_merge.head()


In [0]:

# Estandarización estadística del pronóstico


df_merge["zscorepronostico"] = zscore(
    df_merge["yhat"]
)


In [0]:
# Inspección del resultado con z-score
df_merge.head()


In [0]:
# Verificación de tipos de datos
df_merge.dtypes


In [0]:

# Persistencia del conjunto de predicciones


df_spark_predicciones = spark.createDataFrame(df_merge)

df_spark_predicciones.write \
    .mode("overwrite") \
    .saveAsTable("workspace.gaslp.pred_balance_censo_entregas")


In [0]:

# Comparación general: entregas reales vs pronosticadas


# Copia utilizada exclusivamente para la visualización general.
# No modifica df_merge.
df_grafica_general = (
    df_merge
    .groupby("Fecha", as_index=False)[
        ["entregas_mdb", "yhat", "yhat_lower", "yhat_upper"]
    ]
    .sum()
)

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=df_grafica_general,
    x="Fecha",
    y="entregas_mdb",
    label="Real",
    linewidth=2
)

sns.lineplot(
    data=df_grafica_general,
    x="Fecha",
    y="yhat",
    label="Pronosticado",
    linewidth=2
)

plt.fill_between(
    df_grafica_general["Fecha"],
    df_grafica_general["yhat_lower"],
    df_grafica_general["yhat_upper"],
    alpha=0.2,
    label="Intervalo de confianza"
)

plt.title(
    "Entregas totales: valores reales vs pronóstico Prophet",
    fontsize=14
)
plt.xlabel("Fecha")
plt.ylabel("Entregas (MDB)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [0]:
# Primeras observaciones del conjunto final
df_merge.head()


In [0]:
# Últimas observaciones del conjunto final
df_merge.tail()


In [0]:

# Validación cruzada del modelo Prophet general


validacion = cross_validation(
    m,
    initial="200 days",
    period="90 days",
    horizon="30 days"
)


In [0]:

# Métricas de desempeño


df_p = performance_metrics(validacion)

df_p.head(16)


In [0]:
# Últimos registros de las métricas
df_p.tail(10)


In [0]:

# Visualización de métricas de validación cruzada


fig_rmse = plot_cross_validation_metric(
    validacion,
    metric="rmse"
)
plt.title("Validación cruzada - RMSE")
plt.tight_layout()
plt.show()

fig_mae = plot_cross_validation_metric(
    validacion,
    metric="mae"
)
plt.title("Validación cruzada - MAE")
plt.tight_layout()
plt.show()

fig_coverage = plot_cross_validation_metric(
    validacion,
    metric="coverage"
)
plt.title("Validación cruzada - Coverage")
plt.tight_layout()
plt.show()


In [0]:

# Persistencia de las métricas del modelo


df_spark_metricas = spark.createDataFrame(df_p)

df_spark_metricas.write \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.gaslp.pred_balance_censo_entregas_metricas"
    )
